# WORLD2045 Training Manual Notebook Set-up

## Purpose

This notebook is organized into a clean, repeatable workflow:

1. **Bootstrap**
2. **Data load**
3. **Validation**
4. **Analysis**
5. **Visualization**

It is designed for local DuckDB-based exploration of the World2045 sample parquet extracts.

## Expected local structure

```text
project_root/
├─ data/
│  └─ sample/
│     ├─ country_overrides/
│     │  └─ 000000000000.parquet
│     ├─ dim_country/                  # optional
│     │  └─ 000000000000.parquet
│     ├─ gold__country_rise_potential/
│     │  └─ 000000000000.parquet
│     └─ ...
└─ notebook/
   └─ WORLD2045_Training_Manual_Data_Analyst.ipynb
```

The bootstrap logic resolves the repository root automatically and registers one parquet-backed DuckDB view per folder under `data/sample/`.

Notes:
- Sample folders contain parquet files.
- In your current setup, there is one parquet file per folder.
- The notebook supports either `dim_country` or `dim__country` naming.


## 1. Bootstrap

This cell establishes a robust local DuckDB session, auto-discovers the repository root, registers parquet-backed views, and defines reusable helper functions.

<span style="color:red;"><strong>Important:</strong> Ensure that you have duckdb, pandas, pyarrow, ipykernel and matplotlib installed in your environment.</span>

If not, please install, via terminal, before running the boostrap:

```Bash
    pip install duckdb pandas==2.2.0 pyarrow matplotlib ipykernel
    python -m ipykernel install --user --name world2045 --display-name "Python (world2045)"
```

Run this cell first after every kernel restart.


In [ ]:
import duckdb
import pandas as pd
import pathlib
from typing import Iterable
from IPython.display import display

def find_repo_root(start: pathlib.Path | None = None) -> pathlib.Path:
    start = (start or pathlib.Path.cwd()).resolve()
    for base in [start, *start.parents]:
        if (base / "data" / "sample").exists():
            return base
    raise FileNotFoundError(
        "Could not find repo root. Expected 'data/sample' in the current working directory or one of its parents."
    )

REPO_ROOT = find_repo_root()
SAMPLE_DIR = REPO_ROOT / "data" / "sample"
DB_PATH = REPO_ROOT / "world2045.duckdb"

def connect_world2045(db_path: pathlib.Path = DB_PATH) -> duckdb.DuckDBPyConnection:
    connection = duckdb.connect(str(db_path))
    connection.execute("PRAGMA threads=4")
    connection.execute("PRAGMA enable_progress_bar")
    return connection

con = connect_world2045()

def list_sample_tables(sample_dir: pathlib.Path = SAMPLE_DIR) -> list[str]:
    if not sample_dir.exists():
        raise FileNotFoundError(f"Sample directory not found: {sample_dir}")
    return sorted([p.name for p in sample_dir.iterdir() if p.is_dir()])

def resolve_parquet_source(table_name: str, sample_dir: pathlib.Path = SAMPLE_DIR) -> str:
    table_dir = sample_dir / table_name
    parquet_files = sorted(table_dir.glob("*.parquet"))
    if not parquet_files:
        raise FileNotFoundError(f"No parquet files found in: {table_dir}")
    if len(parquet_files) == 1:
        return parquet_files[0].as_posix()
    return (table_dir / "*.parquet").as_posix()

def register_parquet_views(connection: duckdb.DuckDBPyConnection, table_names: Iterable[str]) -> None:
    for table_name in table_names:
        parquet_source = resolve_parquet_source(table_name)
        connection.execute(
            f'CREATE OR REPLACE VIEW "{table_name}" AS SELECT * FROM read_parquet(\'{parquet_source}\')'
        )

tables = list_sample_tables()
register_parquet_views(con, tables)

# Optional compatibility alias: support dim_country and dim__country interchangeably
if "dim_country" in tables and "dim__country" not in tables:
    con.execute('CREATE OR REPLACE VIEW "dim__country" AS SELECT * FROM "dim_country"')
elif "dim__country" in tables and "dim_country" not in tables:
    con.execute('CREATE OR REPLACE VIEW "dim_country" AS SELECT * FROM "dim__country"')

def run_query(sql: str, preview: int | None = None, quiet: bool = False) -> pd.DataFrame:
    df = con.execute(sql).df()
    if not quiet:
        print(f"Returned {len(df):,} rows x {len(df.columns)} columns")
    if preview is not None:
        display(df.head(preview))
    return df

def show_tables() -> pd.DataFrame:
    return con.execute(
        '''
        select table_name
        from information_schema.tables
        where table_schema = 'main'
        order by table_name
        '''
    ).df()

def describe_table(table_name: str) -> pd.DataFrame:
    return con.execute(f'DESCRIBE "{table_name}"').df()



def _currency_formatter(value):
    if pd.isna(value):
        return ""
    return f"${value:,.2f}"

def _percent_formatter(value):
    if pd.isna(value):
        return ""
    return f"{value:,.2f}%"

def _count_formatter(value):
    if pd.isna(value):
        return ""
    return f"{value:,.0f}"

def _bool_formatter(value):
    if pd.isna(value):
        return ""
    return "Yes" if bool(value) else "No"

def style_dataframe(df):
    usd_columns = [col for col in df.columns if "usd" in col.lower() or "dollar" in col.lower()]
    pct_columns = [
        col for col in df.columns
        if col not in usd_columns and (
            col.lower().endswith("_pct")
            or "percent" in col.lower()
            or "percentage" in col.lower()
            or "pct" in col.lower()
        )
    ]
    count_columns = [
        col for col in df.columns
        if col not in usd_columns + pct_columns and (
            "population" in col.lower()
        )
    ]
    bool_columns = [col for col in df.columns if pd.api.types.is_bool_dtype(df[col])]

    formatters = {col: _currency_formatter for col in usd_columns}
    formatters.update({col: _percent_formatter for col in pct_columns})
    formatters.update({col: _count_formatter for col in count_columns})
    formatters.update({col: _bool_formatter for col in bool_columns})

    styler = (
        df.style
        .set_properties(**{"text-align": "left"})
        .set_table_styles(
            [
                {"selector": "th", "props": [("text-align", "left")]},
                {"selector": "td", "props": [("text-align", "left")]},
            ]
        )
        .format(formatters, na_rep="")
    )
    return styler

def display_df(df: pd.DataFrame, preview: int | None = None):
    to_show = df.head(preview) if preview is not None else df
    display(style_dataframe(to_show))

print(f"Working directory: {pathlib.Path.cwd()}")
print(f"Repo root: {REPO_ROOT}")
print(f"Sample dir: {SAMPLE_DIR}")
print(f"DuckDB path: {DB_PATH}")
print(f"Registered {len(show_tables()):,} parquet-backed views from {SAMPLE_DIR}")


## 2. Data load checks

This section confirms that the expected tables are present before analysis starts.


In [ ]:
REQUIRED_TABLES = [
    "country_overrides",
    "gold__mart_world2045_features_country_year",
    "gold__mart_world2045_features_analytic_1960_2023",
    "gold__features_world2045_normalized_country_year",
    "gold__forecast_feature_country_year",
    "gold__country_trajectory_score_year_scenario",
    "gold__region_trajectory_score_year",
    "gold__subregion_trajectory_score_year",
    "gold__country_rise_potential",
    "gold__country_trajectory_momentum",
    "gold__trajectory_country_quadrant",
]

OPTIONAL_TABLE_ALIASES = [
    ("dim_country", "dim__country"),
]

registered_table_names = show_tables()["table_name"].tolist()
missing_required = [t for t in REQUIRED_TABLES if t not in tables]

missing_optional_alias_groups = []
for left, right in OPTIONAL_TABLE_ALIASES:
    if left not in registered_table_names and right not in registered_table_names:
        missing_optional_alias_groups.append(f"{left} / {right}")

if missing_required:
    raise FileNotFoundError(
        "Missing required sample folders under data/sample: " + ", ".join(missing_required)
    )

if missing_optional_alias_groups:
    print("Optional sample folders not found:", ", ".join(missing_optional_alias_groups))
else:
    print("Optional sample folders are available.")

print("Required table check passed.")


## 3. Table browser

Use these quick inspection outputs before writing or modifying SQL.


In [ ]:
print("Table browser ready.")
print("Available tables:")
_ = display_df(show_tables())

# TRAINING MANUAL

## Project framing

This manual is for a new data analyst joining the World2045 project. It explains:

- what the project is trying to answer
- why the warehouse and analytical layers were designed the way they were
- how to validate intermediate and final outputs
- which SQL commands support key findings

## Core question

**Which countries and regions are structurally strongest by 2045, which are improving fastest, and which remain at structural risk?**

## Layer logic

### Bronze
Preserves raw source structure so ingestion remains auditable.

### Silver
Standardizes country-year facts so different datasets can be aligned.

### Gold
Produces wide feature marts and analytical models that are easy to query and visualize.

## Most important tables

- `gold__mart_world2045_features_country_year`
- `gold__forecast_feature_country_year`
- `gold__country_trajectory_score_year_scenario`
- `gold__country_rise_potential`
- `gold__country_trajectory_momentum`
- `gold__trajectory_country_quadrant`

## Scoring rationale

The final trajectory score combines six conceptual dimensions:

- income strength: GDP per capita
- population wellbeing: life expectancy
- institutions: governance
- climate resilience: adaptation readiness
- climate exposure: vulnerability penalty
- conflict burden: conflict penalty

## Why 1995–2023 for historical scoring

Climate indicators begin in 1995. Because the final score depends on climate variables, the comparable historical scoring window starts there.

## Why forecast scoring is scenario-based

Only some variables have direct forward projections. The current baseline scenario projects GDP per capita and life expectancy directly, while carrying forward latest available historical values for structural variables that do not yet have forecast series.

## SQL workflow

The cells below are grouped by analytical stage:
- source and feature coverage validation
- forecast feature validation
- historical trajectory validation
- forecast completeness validation
- region and subregion validation
- rise-potential validation
- momentum validation
- quadrant validation


In [ ]:
sql = """
SELECT table_name
FROM information_schema.tables
WHERE table_name LIKE '%country%'
ORDER BY table_name;
"""
_ = display_df(run_query(sql, quiet=True))

### A2. Inspect feature mart columns

Used to identify exact field names before writing the scenario model.

In [ ]:
sql = """
select
  table_name,
  column_name,
  data_type
from information_schema.columns
where table_name in (
  'gold__features_world2045_normalized_country_year',
  'gold__mart_world2045_features_analytic_1960_2023',
  'gold__mart_world2045_features_country_year'
)
order by table_name, ordinal_position;
"""
_ = display_df(run_query(sql, quiet=True))

### A3. Inspect country mapping columns

Used to find the correct regional metadata fields.

In [ ]:
sql = '''
select
  table_name,
  column_name,
  data_type
from information_schema.columns
where table_name in (
  'dim_country',
  'dim__country',
  'country_overrides'
)
order by table_name, ordinal_position;
'''
_ = display_df(run_query(sql, quiet=True))

### A4. Preview feature marts

Used to validate that candidate marts actually contain the needed fields.

In [ ]:
sql = """
select *
from gold__mart_world2045_features_country_year
limit 5;
"""
_ = display_df(run_query(sql, quiet=True))

In [ ]:
sql = """
select *
from gold__mart_world2045_features_analytic_1960_2023
limit 5;
"""
_ = display_df(run_query(sql, quiet=True))

In [ ]:
sql = """
select *
from gold__features_world2045_normalized_country_year
limit 5;
"""
_ = display_df(run_query(sql, quiet=True))

---

## B. Forecast feature validation

### B1. Check forecast years and row coverage

Used after building forecast tables.

In [ ]:
sql = """
select
  min(year) as min_year,
  max(year) as max_year,
  count(*) as row_count,
  count(distinct country_iso3) as country_count
from gold__forecast_feature_country_year;
"""
_ = display_df(run_query(sql, quiet=True))

### B2. Null coverage for core forecast fields

In [ ]:
sql = """
select
  count(*) as rows_total,
  count(*) filter (where fertility_rate is null) as fertility_nulls,
  count(*) filter (where life_expectancy_birth_both is null) as life_expectancy_nulls,
  count(*) filter (where net_migrants_thousands is null) as migration_nulls
from gold__forecast_feature_country_year;
"""
_ = display_df(run_query(sql, quiet=True))

---

## C. Historical trajectory validation

### C1. Scenario / score coverage check

In [ ]:
sql = """
select
  scenario_id,
  min(year) as min_year,
  max(year) as max_year,
  count(*) as row_count,
  count(distinct country_iso3) as country_count
from gold__country_trajectory_score_year_scenario
group by 1
order by 1;
"""
_ = display_df(run_query(sql, quiet=True))

### C2. Forecast null leakage check

In [ ]:
sql = """
select
  count(*) as total_rows,
  count(*) filter (where trajectory_score is null) as null_trajectory_score_rows,
  count(*) filter (where gdp_pc_norm is null) as null_gdp_norm_rows,
  count(*) filter (where life_expectancy_norm is null) as null_life_expectancy_norm_rows,
  count(*) filter (where governance_norm is null) as null_governance_norm_rows,
  count(*) filter (where adaptation_readiness_norm is null) as null_adaptation_norm_rows,
  count(*) filter (where climate_vulnerability_norm is null) as null_climate_norm_rows,
  count(*) filter (where conflict_severity_norm is null) as null_conflict_norm_rows
from gold__country_trajectory_score_year_scenario
where is_forecast_year = true;
"""
_ = display_df(run_query(sql, quiet=True))

### C3. 2023 vs 2045 country comparison

Used to inspect major movers.

In [ ]:
sql = """
with base as (
  select
    country_iso3,
    year,
    trajectory_score
  from gold__country_trajectory_score_year_scenario
  where scenario_id in ('historical_observed', 'baseline_static_risk')
    and year in (2023, 2045)
),
pivoted as (
  select
    country_iso3,
    max(case when year = 2023 then trajectory_score end) as score_2023,
    max(case when year = 2045 then trajectory_score end) as score_2045
  from base
  group by 1
)
select
  country_iso3,
  score_2023,
  score_2045,
  score_2045 - score_2023 as score_change_2023_2045
from pivoted
order by score_change_2023_2045 desc
limit 50;
"""
_ = display_df(run_query(sql, quiet=True))

---

## D. Forecast completeness validation

### D1. Forecast completeness counts

Used to determine how many forecast cases were complete vs partial.

In [ ]:
sql = """
select
  case
    when vdem_liberal_democracy_index is not null
     and adaptation_readiness is not null
     and climate_vulnerability is not null
     and conflict_severity is not null
    then 'complete'
    else 'partial'
  end as forecast_score_completeness,
  count(distinct country_iso3) as country_count
from gold__country_trajectory_score_year_scenario
where is_forecast_year = true
group by 1
order by 1;
"""
_ = display_df(run_query(sql, quiet=True))

### D2. Coverage diagnosis for partial rows

Used to identify which countries had missing carried-forward components.

In [ ]:
sql = """
select
  country_iso3,
  count(*) as forecast_rows,
  max(case when vdem_liberal_democracy_index is null then 1 else 0 end) as governance_missing,
  max(case when adaptation_readiness is null then 1 else 0 end) as adaptation_missing,
  max(case when climate_vulnerability is null then 1 else 0 end) as climate_missing,
  max(case when conflict_severity is null then 1 else 0 end) as conflict_missing
from gold__country_trajectory_score_year_scenario
where is_forecast_year = true
  and (
    vdem_liberal_democracy_index is null
    or adaptation_readiness is null
    or climate_vulnerability is null
    or conflict_severity is null
  )
group by 1
order by forecast_rows desc, country_iso3;
"""
_ = display_df(run_query(sql, quiet=True))

# Expected result none.

### D3. ISO mapping gaps for regional analysis

Used to identify unmatched forecast ISO3 values.

In [ ]:
sql = """
select
  s.country_iso3
from gold__country_trajectory_score_year_scenario s
left join country_overrides d
  on s.country_iso3 = d.iso3
where s.year = 2045
  and s.scenario_id = 'baseline_static_risk'
  and d.iso3 is null
group by 1
order by 1;
"""
_ = display_df(run_query(sql, quiet=True))

# Expected result none.

---

## E. Region and subregion validation

### E1. Region-level spot check

Used to inspect a few anchor years.

In [ ]:
sql = """
select
  region,
  year,
  scenario_id,
  avg_trajectory_score,
  country_count
from gold__region_trajectory_score_year
where year in (1995, 2023, 2045)
order by region, year;
"""
_ = display_df(run_query(sql, quiet=True))

### E2. Top subregions by 2045 score

Used to support regional findings.

In [ ]:
sql = """
select
  region,
  subregion,
  avg_trajectory_score,
  country_count
from gold__subregion_trajectory_score_year
where year = 2045
  and scenario_id = 'baseline_static_risk'
order by avg_trajectory_score desc
limit 25;
"""
_ = display_df(run_query(sql, quiet=True))

---

## F. Rise potential validation

### F1. Coverage by completeness

In [ ]:
sql = """
select
  forecast_score_completeness,
  count(*) as row_count,
  count(*) filter (where is_rankable_forecast_case) as rankable_count
from gold__country_rise_potential
group by 1
order by 1;
"""
_ = display_df(run_query(sql, quiet=True))

### F2. Top 25 rise-potential countries

In [ ]:
sql = """
select
  country_iso3,
  country_name,
  region,
  subregion,
  trajectory_score_2023,
  trajectory_score_2045,
  trajectory_change_2023_2045,
  rise_potential_score
from gold__country_rise_potential
where is_rankable_forecast_case = true
  and is_sovereign = true
order by rise_potential_score desc
limit 25;
"""
_ = display_df(run_query(sql, quiet=True))

### F3. Top 10 by region

In [ ]:
sql = """
with ranked as (
  select
    *,
    row_number() over (
      partition by region
      order by rise_potential_score desc
    ) as rn
  from gold__country_rise_potential
  where is_rankable_forecast_case = true
    and is_sovereign = true
    and region is not null
)
select
  region,
  country_iso3,
  country_name,
  rise_potential_score,
  trajectory_change_2023_2045
from ranked
where rn <= 10
order by region, rn;
"""
_ = display_df(run_query(sql, quiet=True))

---

## G. Momentum validation

### G1. Top 25 momentum countries

In [ ]:
sql = """
select
  country_iso3,
  country_name,
  region,
  subregion,
  trajectory_score_2023,
  trajectory_score_2045,
  trajectory_change_2023_2045,
  momentum_score,
  momentum_band
from gold__country_trajectory_momentum
where is_rankable_forecast_case = true
  and is_sovereign = true
order by momentum_score desc
limit 25;
"""
_ = display_df(run_query(sql, quiet=True))

### G2. Top 10 momentum countries by region

In [ ]:
sql = """
with ranked as (
  select
    *,
    row_number() over (
      partition by region
      order by momentum_score desc
    ) as rn
  from gold__country_trajectory_momentum
  where is_rankable_forecast_case = true
    and is_sovereign = true
    and region is not null
)
select
  region,
  country_iso3,
  country_name,
  momentum_score,
  trajectory_change_2023_2045,
  momentum_band
from ranked
where rn <= 10
order by region, rn;
"""
_ = display_df(run_query(sql, quiet=True))

### G3. Distribution by momentum band

In [ ]:
sql = """
select
  momentum_band,
  count(*) as country_count
from gold__country_trajectory_momentum
where is_rankable_forecast_case = true
  and is_sovereign = true
group by 1
order by
  case momentum_band
    when 'very_high' then 1
    when 'high' then 2
    when 'moderate' then 3
    when 'low_positive' then 4
    when 'negative' then 5
    else 6
  end;
"""
_ = display_df(run_query(sql, quiet=True))

---

## H. Quadrant validation

### H1. Quadrant distribution

In [ ]:
sql = """
select
  quadrant_label,
  count(*) as country_count
from gold__trajectory_country_quadrant
where is_rankable_forecast_case = true
  and is_sovereign = true
group by 1
order by country_count desc;
"""
_ = display_df(run_query(sql, quiet=True))

### H2. Top countries in each quadrant

In [ ]:
sql = """
with ranked as (
  select
    *,
    row_number() over (
      partition by quadrant_label
      order by momentum_score desc, trajectory_score_2045 desc
    ) as rn
  from gold__trajectory_country_quadrant
  where is_rankable_forecast_case = true
    and is_sovereign = true
)
select
  quadrant_label,
  country_iso3,
  country_name,
  region,
  trajectory_score_2045,
  momentum_score,
  trajectory_change_2023_2045
from ranked
where rn <= 10
order by quadrant_label, rn;
"""
_ = display_df(run_query(sql, quiet=True))

### H3. Regional quadrant mix

In [ ]:
sql = """
select
  region,
  quadrant_label,
  count(*) as country_count
from gold__trajectory_country_quadrant
where is_rankable_forecast_case = true
  and is_sovereign = true
group by 1,2
order by region, quadrant_label;
"""
_ = display_df(run_query(sql, quiet=True))

## 8. How the main findings were reasoned

### Example: "Guyana is the standout upward mover"
Reasoning path:

1. run top 25 rise-potential query
2. run top 25 momentum query
3. confirm Guyana appears at the top of both or near top of both
4. inspect `trajectory_change_2023_2045`

### Example: "Europe dominates upper-tier projected outcomes"
Reasoning path:

1. run top subregion 2045 query
2. run region spot check
3. run quadrant regional mix query
4. compare Europe against Africa, Americas, and Asia

### Example: "Africa remains concentrated in structural risk"
Reasoning path:

1. run regional quadrant mix query
2. confirm Africa has the largest count in `Structural Risk`
3. compare challenger count and future-leader count

## 9. Final learning guidance

When validating a model, always ask:

1. Does the row coverage look correct?
2. Does the year range look correct?
3. Are nulls concentrated in specific countries or whole regions?
4. Do rankings make structural sense?
5. Are country classifications driven by level, momentum, or both?

This project is strongest when interpreted as a transparent, layered analytical system rather than as a black-box forecast engine.

---

# 4. Visualization

These cells convert validated query outputs into quick analyst-ready charts. They are intentionally lightweight and rely on the already-validated gold tables.

Run these only after the earlier validation and analysis cells succeed.


In [ ]:
# --- Visualization: Top 10 Countries in 2045 by Trajectory Score ---

sql_top_2045 = """
select
  country_iso3,
  trajectory_score
from gold__country_trajectory_score_year_scenario
where year = 2045
order by trajectory_score desc
limit 10
"""

df_top_2045 = run_query(sql_top_2045, quiet=True)

plot_bar(
    df_top_2045,
    x="country_iso3",
    y="trajectory_score",
    title="Top 10 Countries in 2045 by Trajectory Score"
)



### V1. Momentum band distribution

In [ ]:
momentum_band_df = run_query("""
select
  momentum_band,
  count(*) as country_count
from gold__country_trajectory_momentum
where is_rankable_forecast_case = true
  and is_sovereign = true
group by 1
order by
  case momentum_band
    when 'very_high' then 1
    when 'high' then 2
    when 'moderate' then 3
    when 'low_positive' then 4
    when 'negative' then 5
    else 6
  end;
""", quiet=True)
plot_bar(momentum_band_df, x="momentum_band", y="country_count", title="Rankable sovereign countries by momentum band", rot=20)

### V2. Quadrant distribution

In [ ]:
quadrant_df = run_query("""
select
  quadrant_label,
  count(*) as country_count
from gold__trajectory_country_quadrant
where is_rankable_forecast_case = true
  and is_sovereign = true
group by 1
order by country_count desc;
""", quiet=True)
plot_bar(quadrant_df, x="quadrant_label", y="country_count", title="Country distribution across trajectory quadrants", rot=20)

### V3. Top 15 rise-potential countries

In [ ]:
rise_top15_df = run_query("""
select
  country_name,
  rise_potential_score
from gold__country_rise_potential
where is_rankable_forecast_case = true
  and is_sovereign = true
order by rise_potential_score desc
limit 15;
""", quiet=True)
plot_bar(rise_top15_df, x="country_name", y="rise_potential_score", title="Top 15 rise-potential countries", rot=60)